In [0]:
df = spark.table("titanic")

In [0]:
df.columns

In [0]:
df. select("Name", "Age", "Sex").show()

In [0]:
df.filter(df.Survived ==1).show()

In [0]:
df.filter((df.Sex == "female") & (df.Pclass ==1)).show()

In [0]:
df = spark.table("workspace.default.titanic")
display(df)

In [0]:
median_age = df.approxQuantile("Age", [0.5], 0.01)[0]
df = df.fillna({"Age": median_age})

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, split, when, substring

df = df.fillna({"Cabin": "Unknown"})
df = df.withColumn("Cabin_Clean", split(col("Cabin"), " ").getItem(0))

df = df.withColumn(
    "Deck",
    when(col("Cabin_Clean") == "Unknown", "Unknown")
    .otherwise(substring(col("Cabin_Clean"), 1, 1))
)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

In [0]:
from pyspark.sql.functions import col

mode_embarked = df.groupBy("Embarked") \
                  .count() \
                  .orderBy(col("count").desc()) \
                  .first()[0]

print("Mode of Embarked:", mode_embarked)

In [0]:
df = df.fillna({"Embarked": mode_embarked})

In [0]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.titanic_cleaned")

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()
display(spark.table("workspace.default.titanic_cleaned"))

In [0]:
from pyspark.sql.functions import col, sum

df.select(sum(col("Deck").isNull().cast("int")).alias("Deck_nulls")).show()

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("Deck") \
    .saveAsTable("workspace.default.titanic_partitioned")

In [0]:
spark.table("workspace.default.titanic_partitioned").show()

In [0]:
df_cleaned = spark.table("workspace.default.titanic_cleaned")

In [0]:
df_cleaned.printSchema()

In [0]:
from pyspark.sql.functions import col, sum
df_cleaned.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_cleaned.columns]).show()

In [0]:
from pyspark.sql.functions import col, sum, isnan, count

# Load cleaned Delta table
df_cleaned = spark.table("workspace.default.titanic_cleaned")

print("===  Null values check ===")
null_counts = df_cleaned.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_cleaned.columns])
null_counts.show(truncate=False)

print("===  Duplicate rows check ===")
total_rows = df_cleaned.count()
distinct_rows = df_cleaned.dropDuplicates().count()
duplicate_rows = total_rows - distinct_rows
print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicate rows: {duplicate_rows}")

# Check duplicates on PassengerId (should be unique)
print("Duplicate PassengerId (if any):")
df_cleaned.groupBy("PassengerId").count().filter("count > 1").show()

print("===  Double column stats (Age, Fare) ===")
# Summary stats
df_cleaned.select("Age", "Fare").describe().show()

# Check NaN values in doubles
df_cleaned.select([sum(isnan(col(c)).cast("int")).alias(c) for c in ["Age", "Fare"]]).show()

# Check for suspicious negative values
print("Suspicious negative values in Age or Fare:")
df_cleaned.filter((col("Age") < 0) | (col("Fare") < 0)).show()

print("=== 4️⃣ Sample preview ===")
display(df_cleaned.limit(10))  # Databricks display for quick visual check

In [0]:
from pyspark.sql.functions import regexp_extract, trim, regexp_replace, concat_ws, col

# Load cleaned Titanic table (after ETL)
df_cleaned = spark.table("workspace.default.titanic_cleaned")

#Extract Titles
df_cleaned = df_cleaned.withColumn("Title", regexp_extract(col("Name"), r", (\w+)\.", 1))
#Extract Last names
df_cleaned = df_cleaned.withColumn("Last_Name", regexp_extract(col("Name"), r"^(.*?),", 1))
#Extract First name
df_cleaned = df_cleaned.withColumn(
    "First_Name",
    trim(regexp_replace(regexp_extract(col("Name"), r"\. (.*?)(?: \(|$)", 1), r"[()]", ""))
)
#Make new column named: "Full Name Formatted"
df_cleaned = df_cleaned.withColumn(
    "Full_Name_Formatted",
    concat_ws(" ", col("Title"), col("First_Name"), col("Last_Name"))
)
#Show new column
df_cleaned.select("Name", "Title", "First_Name", "Last_Name", "Full_Name_Formatted").show(10, truncate=False)


In [0]:
df_cleaned.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.titanic_cleaned")

In [0]:
df_cleaned = spark.table("workspace.default.titanic_cleaned")
display(df_cleaned)